# Test of puffed inner rim

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import pymcfost as mcfost
import os
import copy
import astropy
from astropy.io import fits
import numpy as np
import scipy
import pandas as pd
from astropy import units as u
import random
import distroi
import sys


sys.path.append(os.path.abspath(".."))  # parent of current working dir
import lib.Katya_func as kf

from typing import Literal, Tuple, Dict, Any

from distroi.auxiliary import constants
from distroi.data import image
from distroi.data import sed
from distroi.model.geom_comp import geom_comp
from distroi.auxiliary import select_data_oifits
from distroi.data.oi_container import OIContainer
from scipy.optimize import minimize_scalar
import scipy.optimize
import scipy.interpolate



    

#modelfolder='/fred/oz061/kandrych/mcfost_test/ar_pup/test_0/'
#pymcfost_dir = "/Users/katerynaandrych/Work/lin/Postdoc/ozstar/mcfost_modelling/iras08544-4431/corporaal_2023_models/pymcfost_test/"
pybads_dir = "/Users/katerynaandrych/Work/lin/Postdoc/ozstar/mcfost_modelling/iras08544-4431/manual_test/puffed_rim/"
os.makedirs(pybads_dir, exist_ok=True)  # no error if it already exists

#interstellar reddening is applied as well on demand, can use different files here
reddening_law_path = '/Users/katerynaandrych/Work/lin/Postdoc/Data/SED_reddening/ISMreddening_law_Cardelli1989.dat'


# folder_sim1="with_puffed_rim"
# folder_sim2="without_puffed_rim"
# simulation_dir1 = pybads_dir+folder_sim1+"/"
# simulation_dir2 = pybads_dir+folder_sim2+"/"

# print (f" parameters in folder {folder_sim}")


# Load original file
pf = kf.ParaFile(pybads_dir+"simulation.para")
# Show current parameters
incl=float(pf.params["imin"])
print(f"Inclination: {incl} deg")
pa=float(pf.params["disk_pa"])
print(f"Position angle: {pa} deg")


/opt/anaconda3/envs/interferometry/lib/python3.12/site-packages/pymcfost/SED.py:10: UserWarning: mpl_scatter_density is not present
  warnings.warn("mpl_scatter_density is not present", UserWarning)


Inclination: 19.0 deg
Position angle: 6.0 deg


In [2]:

#filename of SED catalogue data file
data_filename = '/Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/SED/IRAS08544-4431.phot'
data_wave, data_flux, data_err = kf.load_sed_data(data_filename)
    
# PIONIER data
data_dir_pionier, data_file_pionier = "/Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/PIONIER/", "*.fits"
container_data_pionier = distroi.read_oi_container_from_oifits(data_dir_pionier, data_file_pionier, wave_lims=(1.63, 1.64))

# GRAVITY data
data_dir_gravity, data_file_gravity = "/Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/GRAVITY/", "*1.fits"
container_data_gravity = distroi.read_oi_container_from_oifits(data_dir_gravity, data_file_gravity, wave_lims=(2.199, 2.201))

# VLTI/MATISSE L-band data
data_dir_matisse_l, data_file_matisse_l = "/Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/MATISSE_L/", "*.fits"
container_data_matisse_l = distroi.read_oi_container_from_oifits(data_dir_matisse_l, data_file_matisse_l, wave_lims=(3.48, 3.52))

# VLTI/MATISSE N-band data
data_dir_matisse_n, data_file_matisse_n = "/Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/MATISSE_N/", "*.fits"
container_data_matisse_n = distroi.read_oi_container_from_oifits(data_dir_matisse_n, data_file_matisse_n, wave_lims=(9.9, 10.10), fcorr=True)

data_sed = [data_wave, data_flux, data_err]
data_arrays = [data_sed, container_data_pionier, container_data_gravity, container_data_matisse_l, container_data_matisse_n]






def run_mcfost_chi2(param, keys, data_arg, pybads_dir):
    """
    Run MCFOST for given parameters, calculate chi2 for SED and interferometric data
    Parameters
    ----------
    param : list
        List of parameters to set in the simulation.para file.
    keys:
        which parameters are in param. Names correspond to MCFOST parameter file
    data_arg : list
        List containing data for SED and interferometric observations. First element is SED data (tuple of wavelength, flux, error),
        second element is PIONIER data (OIContainer), third element is GRAVITY data (OIContainer),
        fourth element is MATISSE L-band data (OIContainer), fifth element is MATISSE N-band data (OIContainer).
    pybads_dir
        Location where we should run simulation
    Returns
    -------
    chi_total: float
        Total chi2
    chi2_red_total : float
        Total reduced chi2 value for SED and interferometric data.
    loglike: float
        Joint Log-likelihood for optimisation

    """
    #print(param)
    data_sed = data_arg[0]
    container_data_pionier = data_arg[1]
    container_data_gravity = data_arg[2]
    container_data_matisse_l = data_arg[3]
    container_data_matisse_n = data_arg[4]
    
    pf = kf.ParaFile(pybads_dir+"simulation.para")
    
    folder_sim=""
    for i in range(0,len(keys)):
        folder_sim+=keys[i]+"_"+str(param[i])+"_"
        
    print (f" parameters in folder {folder_sim}")
    
   
     
    simulation_dir = pybads_dir+folder_sim+"/"
    # if not os.path.exists(simulation_dir):
    os.makedirs(simulation_dir, exist_ok=True)  # no error if it already exists

    # Save the modified file
    pf.save(simulation_dir+"simulation.para")
    try:
        #WORKS
        os.chdir(simulation_dir) #this is to change directory to where the simulation.para file is and then run mcfost there
        if param[0]=='with_puffed_rim':
            mcfost.run(simulation_dir+'/simulation.para', options='-puffed_up_rim  10 10 30', delete_previous=True, silent=False)
        else:
            mcfost.run(simulation_dir+'/simulation.para', delete_previous=True, silent=True)

        for wave in [1.63, 2.20, 3.50, 10.0]:
            mcfost.run(simulation_dir+'/simulation.para',options = "-img "+str(wave), delete_previous=False, silent=True)
    except:
        print(f"MCFOST run failed for parameters: {keys} = {param}")
        return 10e6
    # if os.path.exists(simulation_dir):
    #     print('path exists')
    #     if not os.path.exists(simulation_dir+'data_th/sed_rt.fits.gz'):
    #         pf.save(simulation_dir+"simulation.para")

    #         os.chdir(simulation_dir) #this is to change directory to where the simulation.para file is and then run mcfost there
    #         mcfost.run(simulation_dir+'/simulation.para', delete_previous=True, silent=True)
    #     else:
    #         print('SED file exists')
    #     for wave in [1.63, 2.20, 3.50, 10.0]:
    #         if not os.path.exists(simulation_dir+'data_'+str(wave)+'/'+'RT.fits.gz'):
    #             mcfost.run(simulation_dir+'/simulation.para',options = "-img "+str(wave), delete_previous=False, silent=True)
    #         else:
    #             print('Image at '+str(wave)+' micron exists')

        
    chi2_sed, chi2_reduced_sed, loglike_sed= kf.chi2_SED_with_reddening(folder_sim, pybads_dir, data_wave=data_sed[0], data_flux=data_sed[1],data_err=data_sed[2],
                                       plot=True, description=f"{keys} = {param}")
    
    chi2_pionier, chi2_red_pionier, loglike_pionier, num_points_pionier= kf.monochromatic_chi(simulation_dir, img_dir="data_1.63/", container_data=container_data_pionier, vistype='vis2', plot=True, fig_dir=simulation_dir+'figures/', extra_title="PIONIER 1.63", log_plotv=False)
    chi2_gravity, chi2_red_gravity, loglike_gravity, num_points_gravity= kf.monochromatic_chi(simulation_dir, img_dir="data_2.2/", container_data=container_data_gravity, vistype='vis2', plot=True, fig_dir=simulation_dir+'figures/', extra_title="GRAVITY 2.2", log_plotv=False)
    chi2_matisse_l, chi2_red_matisse_l, loglike_matisse_l, num_points_matisse_l= kf.monochromatic_chi(simulation_dir, img_dir="data_3.5/", container_data=container_data_matisse_l,vistype='vis2', plot=True, fig_dir=simulation_dir+'figures/', extra_title="MATISSE L 3.5", log_plotv=True)
    chi2_matisse_n, chi2_red_matisse_n, loglike_matisse_n, num_points_matisse_n= kf.monochromatic_chi(simulation_dir, img_dir="data_10.0/", container_data=container_data_matisse_n, vistype='vis', plot=True, fig_dir=simulation_dir+'figures/', extra_title="MATISSE N 10.0", log_plotv=False)
    
    
    chi_total= chi2_sed + chi2_pionier + chi2_gravity + chi2_matisse_l + chi2_matisse_n
    num_points_total= len(data_sed[0]) + num_points_pionier + num_points_gravity + num_points_matisse_l + num_points_matisse_n
    
    chi2_red_total = chi_total/(num_points_total-5)
    loglike_total=loglike_sed+loglike_pionier+loglike_gravity+loglike_matisse_l+loglike_matisse_n
   
    return  chi_total, chi2_red_total, loglike_total 


2025-10-28 23:47:43,426 - distroi.auxiliary.read_oifits - INFO: Reading from /Users/katerynaandrych/Work/lin/Postdoc/Data/interferometry/IRAS08544-4431/PIONIER/*.fits
2025-10-28 23:47:43,426 - distroi.auxiliary.read_oifits - INFO: Reading IRAS08544-4431_PIONIER_data2015ep.fits...
2025-10-28 23:47:43,427 - distroi.auxiliary.read_oifits - INFO: Reading OI_WAVELENGTH
2025-10-28 23:47:43,433 - distroi.auxiliary.read_oifits - INFO: Reading OI_WAVELENGTH
2025-10-28 23:47:43,434 - distroi.auxiliary.read_oifits - INFO: Reading OI_WAVELENGTH
2025-10-28 23:47:43,435 - distroi.auxiliary.read_oifits - INFO: Reading OI_TARGET
2025-10-28 23:47:43,437 - distroi.auxiliary.read_oifits - INFO: Reading OI_ARRAY
2025-10-28 23:47:43,439 - distroi.auxiliary.read_oifits - INFO: Reading OI_VIS2
2025-10-28 23:47:43,440 - distroi.auxiliary.read_oifits - INFO: Reading OI_VIS2
2025-10-28 23:47:43,442 - distroi.auxiliary.read_oifits - INFO: Reading OI_VIS2
2025-10-28 23:47:43,444 - distroi.auxiliary.read_oifits - 

In [3]:
# Run tests for puffed up rim vs no puffed up rim
#run_mcfost_chi2 here is modifyied to set puffed up rim on or off
#looks like puffed rim does not change much the results

chi_puffed=run_mcfost_chi2(param=['with_puffed_rim'], keys=['mode'], data_arg=data_arrays, pybads_dir=pybads_dir)
chi_not_puffed =run_mcfost_chi2(param=['without_puffed_rim'], keys=['mode'], data_arg=data_arrays, pybads_dir=pybads_dir)

 parameters in folder mode_with_puffed_rim_
pymcfost: Running mcfost ...
 You are running MCFOST 4.1.10
 Git SHA = 3723b76634b5373c7741f191fd7586818d63db5c
 it can be turned back on with -rt2
 Input file read successfully
 Thermal equilibrium calculation
 Temperature calculation under LTE approximation
 Parallelized code on  12 processors
  
Tue Oct 28 23:47:44 AEDT 2025
 Creating directory ././data_th
 Using ray-tracing method 1
 Reading /Users/katerynaandrych/software/mcfost/utils/Dust/sil_amorph_DIANA.lnk
 Number of regions detected: 1
 zone 1 --> region= 1 : R=7.41 to 175.00 AU
 Using        7000 cells
 Total  gas mass in model:  0.100000001      Msun

 grain larger than   524.807495     at R >    130.606781    
 Total dust mass in model:   1.00000005E-03  Msun
 Reference cell is #           1
 Using scattering method 2
 Trying to find appropriate stellar spectra ...
 Star #           1  --> Kurucz7250-1.0.fits.gz (forced)
 Done
 Reading /Users/katerynaandrych/software/mcfost/utils

### Checking MCFOST output header and resolution

In [4]:
 # Convolving with real PSF from observations
star_psf='HD83878'
figfolder_psf='/Users/katerynaandrych/Work/lin/PhD/SPHERE_reduction_data/paper2/mean_combined/'+star_psf+'/'
file_psf=star_psf+'_'+'V'+'_'+'I'+'_meancombined.fits'



results= kf.polarimetric_analysis(pybads_dir+'mode_with_puffed_rim_/', wavelength=1.63, plot=True, fig_dir=None, extra_title="with_puffed_rim", image_scale="asinh",
                                convolution_mode='file', psf_file=file_psf, folder_psf=figfolder_psf,
                                camera='zimpol', unresolved_correction_radius_px=3,
                                background_annulus_mas=(200,250), radial_limit_mas=1000.0,
                                deprojection=(0, 0),
                                azimuthal_r_in_mas=0.0, azimuthal_r_out_mas=100.0, azimuthal_nbins=18, theta0=0.0
                                )

results= kf.polarimetric_analysis(pybads_dir+'mode_without_puffed_rim_/', wavelength=1.63, plot=True, fig_dir=None, extra_title="with_puffed_rim", image_scale="asinh",
                                convolution_mode='file', psf_file=file_psf, folder_psf=figfolder_psf,
                                camera='zimpol', unresolved_correction_radius_px=3,
                                background_annulus_mas=(200,250), radial_limit_mas=1000.0,
                                deprojection=(0, 0),
                                azimuthal_r_in_mas=0.0, azimuthal_r_out_mas=100.0, azimuthal_nbins=18, theta0=0.0
                                )


NameError: name 'kf' is not defined

In [ ]:
#open the required ray-traced SED fits file
hdul=fits.open(simulation_dir+'data_1.63/RT.fits.gz')
print(hdul[0].header)
print('pixel scale '+str(hdul[0].header['CDELT2']*3600*1000)+ ' mas/pixel')
print('max size '+str(hdul[0].header['NAXIS1'])+' pixels')
size_res=hdul[0].header['NAXIS1']*hdul[0].header['CDELT2']*3600*1000
print('max size '+str(size_res)+' mas')
distance = 1220.0  # distance in pc
size_res_au=kf.mas2au(size_res, distance)
print('max size in au '+ str(size_res_au)) #4.44 is distance in kpc


## Polarimetric test

In [ ]:

import subprocess

plt.rc('font',   size=14)          # controls default text sizes
plt.rc('axes',   titlesize=16)     # fontsize of the axes title
plt.rc('axes',   labelsize=16)     # fontsize of the x and y labels
plt.rc('xtick',  labelsize=12)     # fontsize of the tick labels
plt.rc('ytick',  labelsize=12)     # fontsize of the tick labels
plt.rc('legend', fontsize=14)      # legend fontsize
plt.rc('figure', titlesize=16)     # fontsize of the figure title




In [ ]:


fig_dir = simulation_dir+"polarimetric/"
os.makedirs(fig_dir, exist_ok=True)  # no error if it already exists

img_array, header, img_tot, img_q, img_u, img_v, img_star, img_star_sct, img_disk_th, img_disk_th_sct = kf.load_mcfost_images_1wave(simulation_dir, '1.63',ploting=True, save_plots=fig_dir, title_addition=folder_sim)



pi_sum = np.sum(np.sqrt(img_q**2 + img_u**2))
pi_frac = pi_sum/np.sum(img_tot)
print(img_q.shape)
pixel_scale=header['CDELT2']*u.deg.to(u.arcsec)*1000 

print(pixel_scale)
 

In [ ]:
from skimage.transform import rescale, resize, downscale_local_mean
from astropy.convolution import Gaussian2DKernel, convolve, convolve_fft, AiryDisk2DKernel



band='I'
if band=='I':
    ps=3.6
    psf_FWHM=30 #mas
elif band=='V':
    ps=3.6
    psf_FWHM=30 #mas
elif band=='H':
    ps=12.27
    psf_FWHM=40

print('Simulation MCFOST')
R_mcfost, x, y, X, Y=kf.compute_grid(img_q)
#calculating angle for azimuthal polarisation

q_phi, u_phi, pi, phi_mcfost=kf.compute_qphi_uphi_pi(img_q, img_u)

pi_sum = np.sum(pi)
pi_frac = pi_sum/np.sum(img_tot)

print(f'sum Q_phi: {np.sum(q_phi)}, sum U_phi: {np.sum(u_phi)}, sum PI: {pi_sum}, sum I: {np.sum(img_tot)}')
print(f'frac in % Q_phi:{np.sum(q_phi)/np.sum(img_tot)*100}, frac U_phi:{np.sum(u_phi)/np.sum(img_tot)*100}, frac PI: {pi_frac*100}')

print('Rescaling to instrument pixel scale')

img_q_rescaled, img_u_rescaled, img_total_rescaled, q_phi_rescaled, u_phi_rescaled, pi_rescaled, phi_rescaled=kf.rescale_and_recalculate_all_polarim_img(img_q, img_u, img_tot, pixel_scale, instrument='zimpol', conserve='sum')
pi_rescaled_sum = np.sum(pi_rescaled)
pi_rescaled_frac = pi_rescaled_sum/np.sum(img_total_rescaled)
print(f'new shape:{img_q_rescaled.shape}, 1/coefficient: {ps/pixel_scale}')

print(f'sum Q_phi: {np.sum(q_phi_rescaled)}, sum U_phi: {np.sum(u_phi_rescaled)}, sum PI: {pi_rescaled_sum}, sum I: {np.sum(img_total_rescaled)}')
print(f'frac Q_phi:{np.sum(q_phi_rescaled)/np.sum(img_total_rescaled)*100}, frac U_phi:{np.sum(u_phi_rescaled)/np.sum(img_total_rescaled)*100}, frac PI: {pi_rescaled_frac*100}')

kf.plot_polarimetric_image(img_q_rescaled, ps, title='Q Rescaled', roi_half_size=30, image_scale="asinh", save=False, show=True)
kf.plot_polarimetric_image(q_phi_rescaled, ps, title='Q_phi Rescaled', roi_half_size=30, image_scale="asinh", save=False, show=True)



# Convolving with synthetic PSF
kernel, Q_conv, U_conv, I_conv, PI_conv, Q_phi_conv, U_phi_conv=kf.convolve_polarimetric_images(img_q_rescaled, img_u_rescaled, img_total_rescaled, ps,psf_source='synthetic', psf_fwhm_mas=psf_FWHM)
metrics=kf.polarimetric_metrics(I_conv,Q_conv, U_conv, Q_phi_conv, U_phi_conv,PI_conv)
print(metrics)


# Convolving with real PSF from observations
star_psf='HD83878'
figfolder_psf='/Users/katerynaandrych/Work/lin/PhD/SPHERE_reduction_data/paper2/mean_combined/'+star_psf+'/'
file_psf=star_psf+'_'+'V'+'_'+'I'+'_meancombined.fits'

kernel, Q_conv, U_conv, I_conv, PI_conv, Q_phi_conv, U_phi_conv=kf.convolve_polarimetric_images(img_q_rescaled, img_u_rescaled, img_total_rescaled, ps,psf_source='file', psf_file=file_psf, folder_psf=figfolder_psf, psf_cut=100, )

metrics=kf.polarimetric_metrics(I_conv,Q_conv, U_conv, Q_phi_conv, U_phi_conv,PI_conv)
print(metrics)







In [ ]:
kf.plot_polarimetric_image((img_tot-img_star)/np.sum(img_tot), ps, title="tot-star", roi_half_size=50, image_scale='asinh', show=True )
#plot_polarimetric_image(pi_conv, ps, title='PI Conv', roi_half_size=100, image_scale="asinh", save=False, show=True)


In [ ]:
from astropy.io import fits
from skimage.measure import EllipseModel
from matplotlib.patches import Ellipse
from scipy import interpolate
from mpl_toolkits.axes_grid1 import make_axes_locatable
import math
from textwrap import wrap
import scipy.ndimage as ndimage
from matplotlib.gridspec import GridSpec
from matplotlib import colors
import pandas as pd
from scipy.optimize import curve_fit
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties

print(kf.find_FWHM(img_tot, img_tot.shape[0])*pixel_scale)
print(kf.find_FWHM(img_total_rescaled, img_total_rescaled.shape[0])*ps)
print(kf.find_FWHM(I_conv, I_conv.shape[0])*ps)

kf.plot_polarimetric_image(img_tot, pixel_scale, title='I tot')
kf.plot_polarimetric_image(img_total_rescaled, ps, title='I tot rescaled')
kf.plot_polarimetric_image(I_conv, ps, title='I tot conv')

correction_radius=3
R_rescaled,_,_,_,_=kf.compute_grid(img_q_rescaled)

dolp_unres, aolp_unres,q_corr,u_corr=kf.calculate_unresolved(correction_radius, img_q_rescaled, img_u_rescaled,img_total_rescaled,pixel_scale,R_rescaled,100)
q_phi_corr, u_phi_corr, pi_corr, phi =kf.compute_qphi_uphi_pi(q_corr, u_corr)
aolp_corr=0.5*np.arctan2(u_corr, q_corr)

print(f'Unresolved pol: {dolp_unres*100} %, angle: {aolp_unres} deg')

dolp_unres_conv, aolp_unres_conv,q_corr_conv,u_corr_conv=kf.calculate_unresolved(correction_radius, Q_conv, U_conv,I_conv,ps,R_rescaled,100)
q_phi_corr_conv, u_phi_corr_conv, pi_corr_conv, phi =kf.compute_qphi_uphi_pi(q_corr_conv, u_corr_conv)

aolp_corr_conv=0.5*np.arctan2(u_corr_conv, q_corr_conv)
print(f'Unresolved pol after conv: {dolp_unres_conv*100} %, angle: {aolp_unres_conv} deg')




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Optional, Tuple
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable




images_list = [
    q_phi_rescaled, pi_rescaled, q_phi_corr, pi_corr,
    Q_phi_conv, PI_conv, q_phi_corr_conv, pi_corr_conv
]
titles = ['Q$_\\phi$', 'I$_{\\mathrm{pol}}$', 'Q$_\\phi$', 'I$_{\\mathrm{pol}}$',
          'Q$_\\phi$', 'I$_{\\mathrm{pol}}$', 'Q$_\\phi$', 'I$_{\\mathrm{pol}}$']

print(ps, type(ps))

fig, axs = kf.plot_image_grid(
    images=images_list,
    ps_mas=ps,
    nrows=2,
    ncols=4,
    titles=titles,
    group_headers=[(0.31, 'With unresolved'), (0.72, 'Without unresolved')],
    scale="asinh",
    roi_half_size=30,          
    per_panel_autoscale=True,
    colorbar="individual",
    figsize=(12, 6),
    show=True
)
fig.savefig(fig_dir+"mcfost_model_comparison.png", dpi=150, bbox_inches='tight')


In [ ]:

kf.plot_polarimetric_image(q_phi_corr_conv,ps,Q=q_corr_conv,U=u_corr_conv,I=I_conv,title="convolved, unresolved corrected Q phi",bin_factor=(4,4),save=False,snr_threshold=3,noise_level=5e-19,roi_half_size=30,aolp_quiver=True, quiver_scale=0.1)
kf.plot_polarimetric_image(pi_rescaled,ps,Q=img_q_rescaled,U=img_u_rescaled,I=img_total_rescaled,title="Q phi",bin_factor=(4,4),save=False,snr_threshold=3,noise_level=2e-17,roi_half_size=30,aolp_quiver=True, quiver_scale=5)
kf.plot_polarimetric_image(q_phi_corr_conv, ps, roi_half_size=30, image_scale="asinh")
kf.plot_polarimetric_image(Q_conv, ps, roi_half_size=30, image_scale="linear")
kf.plot_polarimetric_image(I_conv, ps, roi_half_size=30, image_scale="linear")

kf.plot_polarimetric_image(q_phi,pixel_scale,Q=img_q,U=img_u,I=img_tot,title="Q phi",bin_factor=(4,4),image_scale="asinh",save=False,snr_threshold=3,noise_level=1e-17,roi_half_size=100,aolp_quiver=True, quiver_scale=3)



## Radial profile

In [ ]:
radial_profile_1=kf.radial_br_profile(PI_conv, ps,incl,pa, R_limit=150, noise_level=0.05,force_stop=True, mode='sum',save=fig_dir+"mcfost_", plot=True,background_annulus_mas=(200,250))

In [ ]:
radial_profile_0=kf.radial_br_profile(PI_conv, ps,0,0,R_limit=150, noise_level=0.05, mode='sum', force_stop=True, save=fig_dir+"mcfost_", plot=True,background_annulus_mas=(200,250))

In [ ]:
PI_conv.shape

In [ ]:
az_profile=kf.azimuthal_profile(PI_conv, ps, r_in_mas=0, r_out_mas=100, plot=True,mode='sum', save=fig_dir+"mcfost_", nbins=10, theta0=0)